# Probe 004 launcher (phase B)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**One-time setup:** create a fine-grained GitHub PAT scoped to this single repository, Contents: Read and write, with an expiry. In Colab: key icon (Secrets) -> add `SCOUT_RESULTS_PAT` -> enable notebook access. The PAT never appears in this notebook or its output.

Results branch (contract-bound): `results/probe-004-8b68640183ee`

Per session: run all cells top to bottom. After a disconnect, rerun all cells -- run.py resumes from the bundle on Drive, and the transport cell pushes whatever is new.

In [1]:
PHASE = 'B'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = 'eb2f5c3192f6098b6938952543426ed4169c9d88'
RESULTS_BRANCH = 'results/probe-004-8b68640183ee'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/004_v2'

In [2]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
GH_PAT = userdata.get('SCOUT_RESULTS_PAT')  # never printed
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # inherited by the run.py child; never printed

Mounted at /content/drive


In [3]:
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

Cloning into '/content/scout-repo'...
remote: Enumerating objects: 2382, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 2382 (delta 4), reused 13 (delta 0), pack-reused 2361 (from 1)
Receiving objects: 100% (2382/2382), 4.17 MiB | 84.00 KiB/s, done.
Resolving deltas: 100% (1589/1589), done.
/content/scout-repo
Note: switching to 'eb2f5c3192f6098b6938952543426ed4169c9d88'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at eb2f5c3 A3 fix: commit th

In [ ]:
!pip install -q -r probes/004/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 156.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78

In [ ]:
!python probes/004/run.py --phase {PHASE} --output-dir {OUTPUT_DIR}

In [ ]:
# E1 transport: mirror the bundle onto the contract-bound results
# branch. The PAT rides in a header, never in argv or output.
import shutil, subprocess, pathlib, base64, datetime
repo = pathlib.Path('/content/scout-repo')
dest = repo / 'probes/004/results_v2'
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, dest)
def git(*a, **k):
    r = subprocess.run(['git', *a], cwd=repo, capture_output=True, text=True, **k)
    if r.returncode: raise SystemExit(f'git {a[0]} failed: {r.stderr[-400:]}')
    return r.stdout
git('config', 'user.email', 'colab-runner@scout.local')
git('config', 'user.name', 'scout colab runner')
auth = base64.b64encode(f'x-access-token:{GH_PAT}'.encode()).decode()
hdr = f'http.extraheader=AUTHORIZATION: basic {auth}'
if subprocess.run(['git', '-c', hdr, 'fetch', 'origin', RESULTS_BRANCH], cwd=repo, capture_output=True).returncode == 0:
    git('checkout', '-B', RESULTS_BRANCH, f'origin/{RESULTS_BRANCH}')
else:
    git('checkout', '-B', RESULTS_BRANCH, PIN_COMMIT)
git('add', '-f', 'probes/004/results_v2')
stamp = datetime.datetime.utcnow().isoformat(timespec='seconds')
subprocess.run(['git', 'commit', '-m', f'session results {stamp}Z'], cwd=repo, capture_output=True)
git('-c', hdr, 'push', 'origin', RESULTS_BRANCH)
print('pushed', RESULTS_BRANCH)

When `run.py` reports the study complete, the results-validate workflow on the pushed branch verifies the bundle and opens the record-result PR. Merging that PR is the human gate.